# **Case-study simulation for the adaptive multimodal biometric fusion framework**

Reproduces the regime comparison and tau/sigma sensitivity analyses reported in Section IV of the paper.

In [1]:
import numpy as np

In [2]:
A_F, A_P = 0.90, 0.85
CHANCE = 0.02
BETA_HIGH, BETA_LOW = (5,1), (1,2)

In [3]:
def pc(a,q):
  return a * q + CHANCE * (1 - q)

In [4]:
def generate_trials(n, p_avail_f, p_avail_p, beta_params ,seed):
  rng = np.random.default_rng(seed)
  avail_f = rng.random(n) < p_avail_f
  avail_p = rng.random(n) < p_avail_p
  a_b, b_b = beta_params
  q_f, q_p = rng.beta(a_b, b_b, n), rng.beta(a_b, b_b, n)
  z_f, z_p = rng.normal(0, 1, n), rng.normal(0, 1, n) #standard normal, scaled by sigma at eval time
  return dict(avail_f=avail_f, avail_p=avail_p, q_f=q_f, q_p=q_p, z_f=z_f, z_p=z_p)

In [5]:
def weighted_result(pcf, pcp, avail_mask_f, avail_mask_p, wf_raw, wp_raw):
  denom = wf_raw + wp_raw
  any_avail = avail_mask_f | avail_mask_p
  degenerate = any_avail & (denom == 0)     #explicit equal-weight tie-break
  wf = np.where(degenerate, np.where(avail_mask_f, 1.0, 0.0), wf_raw)
  wp = np.where(degenerate, np.where(avail_mask_p, 1.0, 0.0), wp_raw)
  denom2 = wf + wp
  empty = (~any_avail) | (denom2 == 0)
  denom_safe = np.where(denom2 == 0, 1e-9, denom2)
  corr = np.where(empty, 0.0, (wf * pcf + wp * pcp) / denom_safe)
  return corr.mean(), empty.mean()

In [6]:
def evaluate(trials, tau, sigma):
  avail_f, avail_p = trials['avail_f'], trials['avail_p']
  q_f, q_p, z_f, z_p = trials['q_f'], trials['q_p'], trials['z_f'], trials['z_p']
  r_f = np.clip(q_f + z_f * sigma, 0, 1)
  r_p = np.clip(q_p + z_p * sigma, 0, 1)
  pcf, pcp = pc(A_F, q_f), pc(A_P, q_p)
  both = avail_f & avail_p

  I = weighted_result(pcf, pcp, both, both, np.where(both, 1.0, 0.0), np.where(both, 1.0, 0.0))
  II = weighted_result(pcf, pcp, both, both, np.where(both, r_f, 0.0), np.where(both, r_p, 0.0))
  III = weighted_result(pcf, pcp, avail_f, avail_p, np.where(avail_f, 1.0, 0.0), np.where(avail_p, 1.0, 0.0))
  IIIW = weighted_result(pcf, pcp, avail_f, avail_p, np.where(avail_f, r_f, 0.0), np.where(avail_p, r_p, 0.0))

  sel_f = np.where(avail_f, r_f, -np.inf) > tau
  sel_p = np.where(avail_p, r_p, -np.inf) > tau

  IV = weighted_result(pcf, pcp, sel_f, sel_p, np.where(sel_f, r_f, 0.0), np.where(sel_p, r_p, 0.0))

  return dict(I=I, II=II, III=III, **{"III-W": IIIW}, IV=IV)


In [7]:
N = 50000
P_AVAIL_P = 0.90
TAU_DEFAULT, SIGMA_DEFAULT = 0.30, 0.15

regimes = {
    "R4": dict(p_avail_f=0.60, beta= BETA_LOW, seed=44), #primary
    "R1": dict(p_avail_f=0.95, beta= BETA_HIGH, seed=41),
    "R2": dict(p_avail_f=0.95, beta= BETA_LOW, seed=42),
    "R3": dict(p_avail_f=0.60, beta= BETA_HIGH, seed=43),
}

In [8]:
print(f"N ={N}/regime. a_f={A_F}, a_p={A_P}, chance={CHANCE}, tau={TAU_DEFAULT}, sigma={SIGMA_DEFAULT}\n")
print(f"{'Regime':14s}{'I':>16s}{'II':>16s}{'III':>16s}{'III-W':>16s}{'IV':>16s}")
trial_cache = {}
for name, cfg in regimes.items():
  trials = generate_trials(N, cfg['p_avail_f'], P_AVAIL_P, cfg['beta'], cfg['seed'])
  trial_cache[name] = trials
  d = evaluate(trials, TAU_DEFAULT, SIGMA_DEFAULT)
  row = "".join(f"{d[k][0]:.3f}({d[k][1]*100:4.1f}%)".rjust(16) for k in ("I", "II", "III", "III-W", "IV"))
  print(f"{name:14s}{row}")

N =50000/regime. a_f=0.9, a_p=0.85, chance=0.02, tau=0.3, sigma=0.15

Regime                       I              II             III           III-W              IV
R4                0.165(46.0%)    0.198(46.0%)    0.290( 4.2%)    0.323( 4.2%)    0.277(37.6%)
R1                0.628(14.2%)    0.638(14.2%)    0.730( 0.5%)    0.740( 0.5%)    0.740( 0.6%)
R2                0.260(14.6%)    0.312(14.6%)    0.303( 0.5%)    0.356( 0.5%)    0.323(28.2%)
R3                0.395(46.0%)    0.401(46.0%)    0.697( 3.8%)    0.703( 3.8%)    0.702( 4.3%)


### Tau sweep on R4

Common random numbers, same trials, only tau varies.

In [9]:
r4 = trial_cache["R4"]
for tau in [0.1, 0.2, 0.3, 0.4, 0.5]:
  d = evaluate(r4, tau, SIGMA_DEFAULT)
  print(f"tau = {tau:.1f}  III-W = {d['III-W'][0]:.3f}(const.) IV={d['IV'][0]:.3f} (abst. {d['IV'][1]*100:4.1f}%)")

tau = 0.1  III-W = 0.323(const.) IV=0.312 (abst. 15.8%)
tau = 0.2  III-W = 0.323(const.) IV=0.300 (abst. 25.5%)
tau = 0.3  III-W = 0.323(const.) IV=0.277 (abst. 37.6%)
tau = 0.4  III-W = 0.323(const.) IV=0.243 (abst. 50.7%)
tau = 0.5  III-W = 0.323(const.) IV=0.198 (abst. 63.4%)


### Sigma sweep on R4

Common random numbers, same trials, only sigma varies.

In [10]:
for sigma in [0.05, 0.15, 0.30, 0.50]:
  d = evaluate(r4, TAU_DEFAULT, sigma)
  print(f"sigma ={sigma:.2f}  III-W = {d['III-W'][0]:.3f}(const.) IV={d['IV'][0]:.3f} (abst. {d['IV'][1]*100:4.1f}%)")

sigma =0.05  III-W = 0.326(const.) IV=0.287 (abst. 39.2%)
sigma =0.15  III-W = 0.323(const.) IV=0.277 (abst. 37.6%)
sigma =0.30  III-W = 0.317(const.) IV=0.256 (abst. 36.3%)
sigma =0.50  III-W = 0.310(const.) IV=0.236 (abst. 36.5%)
